In [31]:
# from pathlib import Path
# import sys

# project_root = Path.cwd().parent  # notebooks -> project root
# sys.path.insert(0, str(project_root))

# from clinical_synopsis.embedder import Embedder
# print("embedder import OK")

# to avoid clinical_synopsis.embedder
from pathlib import Path
import sys

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / "clinical_synopsis"))

from embedder import Embedder
print("embedder import OK")

embedder import OK


# 0. Data in RETRIEVAL folder


# Tring to Compare 3 modes of search did not work

In [33]:
# this doesn't work because of patient ids
test_cases = [
    {
        "patient_id": "PATIENT_001",
        "query": "What oncology-related events are documented?",
        "is_oncology": True,
    },
    {
        "patient_id": "PATIENT_001",
        "query": "What recent conditions are documented?",
        "is_oncology": None,
    },
    {
        "patient_id": "PATIENT_001",
        "query": "What medications are mentioned?",
        "is_oncology": None,
    },
]

In [ ]:
# compare 3 modes of search
# this gives 'I don't know' for all 9 questions for PATIENT_001 (and not all 3 test_cases)
# also note that is judged PARTLY_RELEVANT sometimes!!!!

import pandas as pd
from rag import rag

rows = []

for case in test_cases:
    for search_type in ["lexical", "semantic", "hybrid"]:
        result = rag(
            query=case["query"],
            patient_id=case["patient_id"],
            is_oncology=case["is_oncology"],
            search_type=search_type,
            num_results=5,
        )

        rows.append({
            "patient_id": case["patient_id"],
            "query": case["query"],
            "search_type": search_type,
            "answer": result["answer"],
            "relevance": result.get("relevance"),
            "groundedness": result.get("groundedness"),
            "response_time": result.get("response_time"),
        })

df = pd.DataFrame(rows)
df

,patient_id,query,search_type,answer,relevance,groundedness,response_time
0,PATIENT_001,What oncology-related events are documented?,lexical,I don't know.,NON_RELEVANT,NOT_GROUNDED,3.701555
1,PATIENT_001,What oncology-related events are documented?,semantic,I don't know.,PARTLY_RELEVANT,NOT_GROUNDED,3.295617
2,PATIENT_001,What oncology-related events are documented?,hybrid,I don't know.,NON_RELEVANT,NOT_GROUNDED,2.642054
3,PATIENT_001,What recent conditions are documented?,lexical,I don't know.,NON_RELEVANT,NOT_GROUNDED,2.155872
4,PATIENT_001,What recent conditions are documented?,semantic,I don't know.,NON_RELEVANT,NOT_GROUNDED,2.411471
5,PATIENT_001,What recent conditions are documented?,hybrid,I don't know.,PARTLY_RELEVANT,NOT_GROUNDED,2.915230
6,PATIENT_001,What medications are mentioned?,lexical,I don't know.,PARTLY_RELEVANT,NOT_GROUNDED,3.914935
7,PATIENT_001,What medications are mentioned?,semantic,I don't know.,NON_RELEVANT,NOT_GROUNDED,2.482412
8,PATIENT_001,What medications are mentioned?,hybrid,I don't know.,NON_RELEVANT,NOT_GROUNDED,2.173699



# 1. Inspect retrieval of chunks

Do not only compare final answers. For a few questions, inspect the retrieved chunks directly, because retrieval quality and answer quality are related but not identical in RAG evaluation. This is often the most informative step when deciding whether semantic or hybrid retrieval is actually helping.

What success looks like

In the notebook, you want to answer questions like:

Does semantic search retrieve more conceptually relevant chunks?

Does lexical search do better on exact terms, names, or identifiers?

Does hybrid search recover both kinds of evidence more reliably?

That kind of side-by-side benchmark is exactly how hybrid retrieval is typically justified in practice.



# First rag result is empty
Patient IDs value mismatch apparently

In [35]:
result = rag(
    query="What oncology-related events are documented?",
    patient_id="PATIENT_001",
    is_oncology=True,
    search_type="hybrid",
    num_results=5,
)

for i, doc in enumerate(result["search_results"], start=1):
    print("=" * 80)
    print("Rank:", i)
    print("Doc type:", doc.get("doc_type"))
    print("Title:", doc.get("title"))
    print("Heading:", doc.get("heading"))
    print("Chunk ID:", doc.get("chunk_id"))
    print("Text:", doc.get("chunk_text", "")[:800])

In [36]:
#If len(result["search_results"]) is 0, that explains why nothing printed.
type(result)
# result.keys()
len(result["search_results"])
# result["search_results"]

0

In [37]:
result

{'answer': "I don't know.",
 'model_used': 'gpt-5.4-mini',
 'search_type': 'hybrid',
 'response_time': 2.4917378425598145,
 'relevance': 'NON_RELEVANT',
 'groundedness': 'NOT_GROUNDED',
 'evaluation_explanation': 'The answer does not address the question about documented oncology-related events and provides no information derived from the retrieved context.',
 'search_results': [],
 'prompt_tokens': 98,
 'completion_tokens': 8,
 'total_tokens': 106,
 'eval_prompt_tokens': 222,
 'eval_completion_tokens': 57,
 'eval_total_tokens': 279}

That output is actually very informative: the problem is not the notebook loop. The problem is that your retrieval returned no chunks at all, so there was nothing to print. Your search_results is an empty list, which is why the for loop never ran.

What this tells us
Since search_type was "hybrid", one or both of these is happening:

lexical search returned [],

semantic search returned [],

or both returned [].

Given your setup, the most likely cause is filter mismatch, especially with patient_id or is_oncology. Your minsearch filtering is exact-match equality on keyword fields, so if the filter value does not exactly match what is stored in the index, you get zero results.

Most likely culprit

This part is especially suspicious:

```python
if is_oncology is not None:
    filter_dict["is_oncology"] = str(int(bool(is_oncology)))
```

If your indexed documents stored is_oncology as:

integer 1 / 0, or

boolean True / False

but minsearch is filtering with the string "1" or "0", then lexical search can return nothing because minsearch uses exact equality on keyword fields.

The same issue can happen in semantic filtering if your loaded metadata stores is_oncology differently than your condition expects.

What to test next

Run these notebook checks.

If 1-2 If return documents, your lexical index is fine and the hybrid logic is also basically fine.

In [38]:
# 1. Test lexical search with no filters
result = rag(
    query="oncology",
    search_type="lexical",
    num_results=5
)

len(result["search_results"]), result["search_results"][:1]

(5,
 [{'id': '153ce7037b2c64421198ec86a7634ce2e5c95c93',
   'chunk_id': '153ce7037b2c64421198ec86a7634ce2e5c95c93',
   'patient_id': 'f1e2eedf-23b6-6c61-c069-9752c03bae16',
   'document_id': '4c4dd15cb45e4127835fc5bf7346b2b500792224',
   'doc_type': 'oncology_timeline',
   'title': 'Oncology Timeline: Harrison106 Rogahn59',
   'heading': 'Oncology Timeline: Harrison106 Rogahn59',
   'chunk_text': '- Patient ID: f1e2eedf-23b6-6c61-c069-9752c03bae16\n- Oncology-related dated events: 52',
   'chunk_index': 0,
   'is_oncology': '1',
   'date_start': '',
   'date_end': ''}])

In [39]:
# 2. Test hybrid with no filters
result = rag(
    query="oncology",
    search_type="hybrid",
    num_results=5
)

len(result["search_results"]), result["search_results"][:1]

(5,
 [{'id': '153ce7037b2c64421198ec86a7634ce2e5c95c93',
   'chunk_id': '153ce7037b2c64421198ec86a7634ce2e5c95c93',
   'patient_id': 'f1e2eedf-23b6-6c61-c069-9752c03bae16',
   'document_id': '4c4dd15cb45e4127835fc5bf7346b2b500792224',
   'doc_type': 'oncology_timeline',
   'title': 'Oncology Timeline: Harrison106 Rogahn59',
   'heading': 'Oncology Timeline: Harrison106 Rogahn59',
   'chunk_text': '- Patient ID: f1e2eedf-23b6-6c61-c069-9752c03bae16\n- Oncology-related dated events: 52',
   'chunk_index': 0,
   'is_oncology': '1',
   'date_start': '',
   'date_end': '',
   'rrf_score': 0.01639344262295082}])

In [40]:
# 3. Test patient filter only
result = rag(
    query="oncology",
    patient_id="PATIENT_001",
    search_type="hybrid",
    num_results=5
)

len(result["search_results"]), result["search_results"][:1]

(0, [])

In [41]:
# 4. Test is_oncology separately
result = rag(
    query="oncology",
    is_oncology=True,
    search_type="hybrid",
    num_results=5
)

len(result["search_results"]), result["search_results"][:3]

(5,
 [{'id': '153ce7037b2c64421198ec86a7634ce2e5c95c93',
   'chunk_id': '153ce7037b2c64421198ec86a7634ce2e5c95c93',
   'patient_id': 'f1e2eedf-23b6-6c61-c069-9752c03bae16',
   'document_id': '4c4dd15cb45e4127835fc5bf7346b2b500792224',
   'doc_type': 'oncology_timeline',
   'title': 'Oncology Timeline: Harrison106 Rogahn59',
   'heading': 'Oncology Timeline: Harrison106 Rogahn59',
   'chunk_text': '- Patient ID: f1e2eedf-23b6-6c61-c069-9752c03bae16\n- Oncology-related dated events: 52',
   'chunk_index': 0,
   'is_oncology': '1',
   'date_start': '',
   'date_end': '',
   'rrf_score': 0.01639344262295082},
  {'chunk_id': '5bad52c43002eea0910cb96a3cebfee9902f69f7',
   'patient_id': '0c0f2095-e8ab-7ac4-6ef4-625748255480',
   'document_id': 'fd281ea0415049cf965a22670025d7e3463d9c3a',
   'doc_type': 'oncology_timeline',
   'title': 'Oncology Timeline: Whitley172 Flatley871',
   'heading': 'Timeline',
   'chunk_text': "10T21:25:23-04:00 — Observation: Cancer Disease Progression; detail: Pati

Good — that isolates it. Your retrieval pipeline basically works, and the problem is specifically the patient_id exact-match filter. Exact-match filters only return documents when the stored value matches the query value exactly, so even small formatting differences break the match.

What this means
Because:

no-filter search works,

hybrid with is_oncology works,

but patient_id=... returns [],

the likely issue is one of these:

the patient_id you pass is not the same string as the one stored in the index,

the field was indexed under a slightly different format, such as Patient/123, 123, or a UUID-like value,

or there is whitespace/case formatting mismatch. Exact filters are very sensitive to representation differences.

Best next inspection
Now the most useful thing is to inspect the actual stored patient IDs from both retrieval sources.



In [43]:
# Check vector documents:
# vector_documents[0] # NameError: name 'vector_documents' is not defined

import rag # because earlier we only imported the rag function, not other module globals such as vector_documents
len(rag.vector_documents)
rag.vector_documents[0]


{'chunk_id': 'b0e8255f623e45e061e672433cf23bdb67a8acc9',
 'patient_id': '005f39cb-0a09-d3ce-596e-96011ede6c69',
 'document_id': '27d7a397916015b73cf3132a1fc47f9e409bf966',
 'doc_type': 'encounters',
 'title': 'encounters.csv',
 'heading': 'encounters',
 'chunk_text': 'patient_id: 005f39cb-0a09-d3ce-596e-96011ede6c69; source_file: data/prototype/sample50/Bridgette172_Huel628_005f39cb-0a09-d3ce-596e-96011ede6c69.json; resource_id: 51ab0337-8247-b585-0b6b-cd04f44bb3ef; subject_reference: urn:uuid:005f39cb-0a09-d3ce-596e-96011ede6c69; encounter_class: AMB; encounter_type_system: http://snomed.info/sct; encounter_type_code: 410620009; encounter_type_display: Well child visit (procedure); status: finished; period_start: 1968-11-19T04:25:28-05:00; period_end: 1968-11-19T04:40:28-05:00; service_provider: ALL-ACCESS PHYSICAL THERAPY, INC.',
 'chunk_index': 0,
 'is_oncology': 0,
 'date_start': '1968-11-19T04:25:28-05:00',
 'date_end': '1968-11-19T04:40:28-05:00'}

In [44]:
sorted({doc["patient_id"] for doc in rag.vector_documents})[:20]


['005f39cb-0a09-d3ce-596e-96011ede6c69',
 '03889212-b18b-4c00-72e0-da0e96f7cf68',
 '03b93198-d95e-c385-c3a7-80470f411d18',
 '03ded366-37cf-72f9-41b9-c43530931cb8',
 '0c0f2095-e8ab-7ac4-6ef4-625748255480',
 '102f518b-da64-8e09-75e6-d8b4b457ea06',
 '1217a34b-dd29-6f88-cb5b-2758720b1bf8',
 '1411fbab-9e86-f27f-aa0c-11865e18f0b2',
 '1fd40aad-b39d-bd5c-ea3f-4e2d7d498f0a',
 '20a8f1af-a818-7ef8-dee6-c2cfb25a0c9d',
 '314fee90-61dd-e0f5-31f6-bc3bc139ddd6',
 '3a48cc84-2b5c-1680-98dd-63a292575e1e',
 '3af995f1-02a5-07ee-5a7e-e2470a017f1e',
 '3ec575fb-56b9-7103-f24f-fd947a4112f2',
 '44e04491-a3a9-53f0-9e08-6b2cf5fc314b',
 '51425e8d-a216-e85a-0aa3-8cdd947d5612',
 '568ec0af-94fa-521b-012e-88f61f78028f',
 '6a908e44-c17c-1049-c120-c024eed67c23',
 '71234514-b867-f37a-3f0e-77e06a769224',
 '71b67010-c7e8-3d3d-d56c-b79b59c595d3']

In [45]:
# Check lexical results
from rag import search
search("oncology", num_results=5)
# we see patient id
#   'patient_id': 'f1e2eedf-23b6-6c61-c069-9752c03bae16',

[{'id': '153ce7037b2c64421198ec86a7634ce2e5c95c93',
  'chunk_id': '153ce7037b2c64421198ec86a7634ce2e5c95c93',
  'patient_id': 'f1e2eedf-23b6-6c61-c069-9752c03bae16',
  'document_id': '4c4dd15cb45e4127835fc5bf7346b2b500792224',
  'doc_type': 'oncology_timeline',
  'title': 'Oncology Timeline: Harrison106 Rogahn59',
  'heading': 'Oncology Timeline: Harrison106 Rogahn59',
  'chunk_text': '- Patient ID: f1e2eedf-23b6-6c61-c069-9752c03bae16\n- Oncology-related dated events: 52',
  'chunk_index': 0,
  'is_oncology': '1',
  'date_start': '',
  'date_end': ''},
 {'id': 'a00ed26ee2db5ea2d2fd946616b7ee70930c4f28',
  'chunk_id': 'a00ed26ee2db5ea2d2fd946616b7ee70930c4f28',
  'patient_id': 'c423d1d3-d0b9-aca6-67a7-0976fb02fac1',
  'document_id': '87a00e63a23bb4b0e38c38d7011d928ec492acfb',
  'doc_type': 'oncology_timeline',
  'title': 'Oncology Timeline: Richie600 Bauch723',
  'heading': 'Oncology Timeline: Richie600 Bauch723',
  'chunk_text': '- Patient ID: c423d1d3-d0b9-aca6-67a7-0976fb02fac1\n- O

In [46]:
# get 'patient_id' directly
results = search("oncology", num_results=10)
[(doc.get("patient_id"), doc.get("title")) for doc in results]

[('f1e2eedf-23b6-6c61-c069-9752c03bae16',
  'Oncology Timeline: Harrison106 Rogahn59'),
 ('c423d1d3-d0b9-aca6-67a7-0976fb02fac1',
  'Oncology Timeline: Richie600 Bauch723'),
 ('a8c24bb7-7761-e0dc-ccae-3595bfe08cd4',
  'Oncology Timeline: Felton646 Quitzon246'),
 ('6a908e44-c17c-1049-c120-c024eed67c23',
  'Oncology Timeline: Rolland302 Walter473'),
 ('b73cbc40-ef87-47d5-08dc-4288048dbf1c',
  'Oncology Timeline: Dan465 Pollich983'),
 ('3a48cc84-2b5c-1680-98dd-63a292575e1e',
  'Oncology Timeline: Teresia279 Boyle917'),
 ('eed821bd-0b0a-31b7-14fc-bb305f584a84',
  'Oncology Timeline: Leon728 Gusikowski974'),
 ('c4681aed-f0ca-7373-f88f-d95e7dd026fa',
  'Oncology Timeline: Lee268 Pollich983'),
 ('3af995f1-02a5-07ee-5a7e-e2470a017f1e',
  'Oncology Timeline: Deandrea896 Johns824'),
 ('807323be-063b-60f2-b6e5-a7e8d600ee3c',
  'Oncology Timeline: Shasta644 Yundt842')]

# Fixing Patient ID mismatch

Great — that means both retrieval sources are populated correctly, and the remaining issue is almost certainly just a patient ID value mismatch in the call, not a broken index. Since exact-match filtering depends on the precise stored value, using a patient ID copied directly from either retrieval source should fix the empty-result problem.

What to do next

Now test with a patient ID taken directly from the data you just inspected, for example:

(If that works, then your hybrid pipeline is functioning and the earlier failure was just caused by using the wrong patient ID string.)

In [47]:
import rag as rag

pid = rag.vector_documents[0]["patient_id"]

result = rag.rag(
    query="What oncology-related events are documented?",
    patient_id=pid,
    search_type="hybrid",
    num_results=5,
)

len(result["search_results"]), result["search_results"][:2]

(5,
 [{'id': '35daedec2f411c30f9ab0335398985a59dc02f9f',
   'chunk_id': '35daedec2f411c30f9ab0335398985a59dc02f9f',
   'patient_id': '005f39cb-0a09-d3ce-596e-96011ede6c69',
   'document_id': '3ca829ee4ebc2159d1ad0e145a6fe5a5c3ab91d3',
   'doc_type': 'oncology_timeline',
   'title': 'Oncology Timeline: Bridgette172 Huel628',
   'heading': 'Oncology Timeline: Bridgette172 Huel628',
   'chunk_text': '- Patient ID: 005f39cb-0a09-d3ce-596e-96011ede6c69\n- Oncology-related dated events: 62',
   'chunk_index': 0,
   'is_oncology': '1',
   'date_start': '',
   'date_end': '',
   'rrf_score': 0.01639344262295082},
  {'chunk_id': 'b06c1b4c0a6f123d7a13379374b02504e1381422',
   'patient_id': '005f39cb-0a09-d3ce-596e-96011ede6c69',
   'document_id': 'dd448f3d5ed4bf0510fd84d470b16edab7381037',
   'doc_type': 'oncology_timeline_events',
   'title': 'oncology_timeline_events.csv',
   'heading': 'oncology_timeline_events',
   'chunk_text': 'event_type: Observation; date: 2012-03-16T10:50:48-04:00; labe

In [48]:
for i, doc in enumerate(result["search_results"], start=1):
    print("=" * 80)
    print("Rank:", i)
    print("Chunk ID:", doc.get("chunk_id"))
    print("Patient ID:", doc.get("patient_id"))
    print("Doc type:", doc.get("doc_type"))
    print("Title:", doc.get("title"))
    print("Heading:", doc.get("heading"))
    print("Text:", doc.get("chunk_text", "")[:500])

Rank: 1
Chunk ID: 35daedec2f411c30f9ab0335398985a59dc02f9f
Patient ID: 005f39cb-0a09-d3ce-596e-96011ede6c69
Doc type: oncology_timeline
Title: Oncology Timeline: Bridgette172 Huel628
Heading: Oncology Timeline: Bridgette172 Huel628
Text: - Patient ID: 005f39cb-0a09-d3ce-596e-96011ede6c69
- Oncology-related dated events: 62
Rank: 2
Chunk ID: b06c1b4c0a6f123d7a13379374b02504e1381422
Patient ID: 005f39cb-0a09-d3ce-596e-96011ede6c69
Doc type: oncology_timeline_events
Title: oncology_timeline_events.csv
Heading: oncology_timeline_events
Text: event_type: Observation; date: 2012-03-16T10:50:48-04:00; label: Treatment status Cancer; status: Treatment changed (situation); resource_id: 4214c720-0566-abfd-dfc6-1d4ae71beca5; source_file: data/prototype/sample50/Bridgette172_Huel628_005f39cb-0a09-d3ce-596e-96011ede6c69.json
Rank: 3
Chunk ID: 268f9254c608ffbe62cd54431879dc2e46c74f5b
Patient ID: 005f39cb-0a09-d3ce-596e-96011ede6c69
Doc type: oncology_timeline
Title: Oncology Timeline: Bridgette172 H

# 2. Inspect retrieval of chunks with exact patient IDs

To make notebook testing easier, create a tiny helper cell to copy one exact patient ID from there each time and avoid this mismatch again.

You’re now at the point where it makes sense to compare:

- lexical,

- semantic,

- hybrid

for the same patient and same query. That is the right setup for the notebook experiment before you build the CSV evaluation runner.





In [49]:
available_patient_ids = sorted({doc["patient_id"] for doc in rag.vector_documents})
available_patient_ids[:10]

['005f39cb-0a09-d3ce-596e-96011ede6c69',
 '03889212-b18b-4c00-72e0-da0e96f7cf68',
 '03b93198-d95e-c385-c3a7-80470f411d18',
 '03ded366-37cf-72f9-41b9-c43530931cb8',
 '0c0f2095-e8ab-7ac4-6ef4-625748255480',
 '102f518b-da64-8e09-75e6-d8b4b457ea06',
 '1217a34b-dd29-6f88-cb5b-2758720b1bf8',
 '1411fbab-9e86-f27f-aa0c-11865e18f0b2',
 '1fd40aad-b39d-bd5c-ea3f-4e2d7d498f0a',
 '20a8f1af-a818-7ef8-dee6-c2cfb25a0c9d']

In [50]:
result = rag.rag(
    query="What oncology-related events are documented?",
    patient_id=available_patient_ids[0],
    is_oncology=True,
    search_type="hybrid",
    num_results=5,
)

for i, doc in enumerate(result["search_results"], start=1):
    print("=" * 80)
    print("Rank:", i)
    print("Doc type:", doc.get("doc_type"))
    print("Title:", doc.get("title"))
    print("Heading:", doc.get("heading"))
    print("Chunk ID:", doc.get("chunk_id"))
    print("Text:", doc.get("chunk_text", "")[:800])

Rank: 1
Doc type: oncology_timeline
Title: Oncology Timeline: Bridgette172 Huel628
Heading: Oncology Timeline: Bridgette172 Huel628
Chunk ID: 35daedec2f411c30f9ab0335398985a59dc02f9f
Text: - Patient ID: 005f39cb-0a09-d3ce-596e-96011ede6c69
- Oncology-related dated events: 62
Rank: 2
Doc type: oncology_timeline_events
Title: oncology_timeline_events.csv
Heading: oncology_timeline_events
Chunk ID: b06c1b4c0a6f123d7a13379374b02504e1381422
Text: event_type: Observation; date: 2012-03-16T10:50:48-04:00; label: Treatment status Cancer; status: Treatment changed (situation); resource_id: 4214c720-0566-abfd-dfc6-1d4ae71beca5; source_file: data/prototype/sample50/Bridgette172_Huel628_005f39cb-0a09-d3ce-596e-96011ede6c69.json
Rank: 3
Doc type: oncology_timeline
Title: Oncology Timeline: Bridgette172 Huel628
Heading: Timeline
Chunk ID: 268f9254c608ffbe62cd54431879dc2e46c74f5b
Text: - 1970-11-19T04:25:28-05:00 — Condition: Acute myeloid leukemia, disease (disorder); detail: resolved; resource_id: 

## Important test_cases fix

Your test_cases now need to use real patient IDs from the index, not placeholder IDs. For example:

In [51]:
test_cases = [
    {
        "patient_id": available_patient_ids[0],
        "query": "What oncology-related events are documented?",
        "is_oncology": True,
    },
    {
        "patient_id": available_patient_ids[0],
        "query": "What recent conditions are documented?",
        "is_oncology": None, # so retrieval may pick chunks where is_oncology is 0, 1, or entirely absent.
    },
    {
        "patient_id": available_patient_ids[0],
        "query": "What medications are mentioned?",
        "is_oncology": None,
    },
]

Note above:
Rows for the second and third questions (is_oncology=None) can include any chunks, including ones where the is_oncology field is missing in your underlying data.

In [ ]:
# rows = []

# for case in test_cases:
#     for search_type in ["lexical", "semantic", "hybrid"]:
#         result = rag.rag(
#             query=case["query"],
#             patient_id=case["patient_id"],
#             is_oncology=case["is_oncology"],
#             search_type=search_type,
#             num_results=5,
#         )

#         rows.append({
#             "patient_id": case["patient_id"],
#             "query": case["query"],
#             "search_type": search_type,
#             "answer": result["answer"],
#             "relevance": result.get("relevance"),
#             "groundedness": result.get("groundedness"),
#             "response_time": result.get("response_time"),
#             "n_search_results": len(result.get("search_results", [])),
#         })

In [ ]:
# import pandas as pd
# df = pd.DataFrame(rows)
# df

,patient_id,query,search_type,answer,relevance,groundedness,response_time,n_search_results
0,005f39cb-0a09-d3ce-596e-96011ede6c69,What oncology-related events are documented?,lexical,The oncology timeline documents these key even...,RELEVANT,GROUNDED,4.515679,5
1,005f39cb-0a09-d3ce-596e-96011ede6c69,What oncology-related events are documented?,semantic,The oncology-related events documented in the ...,RELEVANT,GROUNDED,3.113164,5
2,005f39cb-0a09-d3ce-596e-96011ede6c69,What oncology-related events are documented?,hybrid,The oncology-related events documented include...,RELEVANT,PARTLY_GROUNDED,4.975077,5
3,005f39cb-0a09-d3ce-596e-96011ede6c69,What recent conditions are documented?,lexical,Recent conditions documented in the **conditio...,RELEVANT,GROUNDED,3.729879,5
4,005f39cb-0a09-d3ce-596e-96011ede6c69,What recent conditions are documented?,semantic,The recent conditions documented in the **Pati...,RELEVANT,GROUNDED,10.690483,5
5,005f39cb-0a09-d3ce-596e-96011ede6c69,What recent conditions are documented?,hybrid,The recent conditions documented in the **Pati...,RELEVANT,GROUNDED,3.863995,5
6,005f39cb-0a09-d3ce-596e-96011ede6c69,What medications are mentioned?,lexical,The medications mentioned in the **medications...,RELEVANT,GROUNDED,2.664390,5
7,005f39cb-0a09-d3ce-596e-96011ede6c69,What medications are mentioned?,semantic,The medications mentioned are:\n\n- Acetaminop...,RELEVANT,GROUNDED,5.183838,5
8,005f39cb-0a09-d3ce-596e-96011ede6c69,What medications are mentioned?,hybrid,The medications mentioned in the record are:\n...,RELEVANT,GROUNDED,4.841855,5


A df or more debugging, so you can see:

- whether retrieval returned anything,

- what kind of document came first,

- and whether the answer quality changes across search types.

In [ ]:
# this takes 42s

rows = []

for case in test_cases:
    for search_type in ["lexical", "semantic", "hybrid"]:
        result = rag.rag(
            query=case["query"],
            patient_id=case["patient_id"],
            is_oncology=case["is_oncology"],
            search_type=search_type,
            num_results=5,
        )

        rows.append({
            "patient_id": case["patient_id"],
            "query": case["query"],
            "is_oncology": case["is_oncology"],
            "search_type": search_type,
            "answer": result["answer"],
            "relevance": result.get("relevance"),
            "groundedness": result.get("groundedness"),
            "response_time": result.get("response_time"),
            "n_search_results": len(result.get("search_results", [])),
            "top_doc_type": result["search_results"][0].get("doc_type") if result.get("search_results") else None,
            "top_title": result["search_results"][0].get("title") if result.get("search_results") else None,
        })

df = pd.DataFrame(rows)
df

,patient_id,query,is_oncology,search_type,answer,relevance,groundedness,response_time,n_search_results,top_doc_type,top_title
0,005f39cb-0a09-d3ce-596e-96011ede6c69,What oncology-related events are documented?,True,lexical,Oncology-related events documented in the **on...,RELEVANT,PARTLY_GROUNDED,6.711428,5,oncology_timeline,Oncology Timeline: Bridgette172 Huel628
1,005f39cb-0a09-d3ce-596e-96011ede6c69,What oncology-related events are documented?,True,semantic,The oncology-related events documented in the ...,RELEVANT,GROUNDED,3.766422,5,oncology_timeline_events,oncology_timeline_events.csv
2,005f39cb-0a09-d3ce-596e-96011ede6c69,What oncology-related events are documented?,True,hybrid,Oncology-related events documented for this pa...,RELEVANT,PARTLY_GROUNDED,7.152036,5,oncology_timeline,Oncology Timeline: Bridgette172 Huel628
3,005f39cb-0a09-d3ce-596e-96011ede6c69,What recent conditions are documented?,None,lexical,The recent documented conditions in the **cond...,RELEVANT,GROUNDED,4.026008,5,conditions,conditions.csv
4,005f39cb-0a09-d3ce-596e-96011ede6c69,What recent conditions are documented?,None,semantic,The **recent conditions** documented in the **...,RELEVANT,GROUNDED,4.267335,5,patient_overview,Patient Overview: Bridgette172 Huel628
5,005f39cb-0a09-d3ce-596e-96011ede6c69,What recent conditions are documented?,None,hybrid,Recent conditions documented in the **Patient ...,RELEVANT,GROUNDED,4.507259,5,conditions,conditions.csv
6,005f39cb-0a09-d3ce-596e-96011ede6c69,What medications are mentioned?,None,lexical,The medications mentioned in the **medications...,RELEVANT,GROUNDED,3.377670,5,medications,medications.csv
7,005f39cb-0a09-d3ce-596e-96011ede6c69,What medications are mentioned?,None,semantic,The medications mentioned in the record are:\n...,RELEVANT,GROUNDED,3.537058,5,patient_overview,Patient Overview: Bridgette172 Huel628
8,005f39cb-0a09-d3ce-596e-96011ede6c69,What medications are mentioned?,None,hybrid,The medications mentioned in the record includ...,RELEVANT,GROUNDED,4.651533,5,medications,medications.csv


Because you already collect answer_data into a dataframe, you’ll be able to do:

```python
df["total_cost_usd"].sum()
df.groupby("search_type")["total_cost_usd"].sum()
```
to answer questions like:

“How much did it cost to answer and evaluate these questions for one patient?”

“Does hybrid search cost more than lexical for this experiment, and by how much?”

All of that comes from the simple per‑million pricing you just confirmed.

Input: $0.75 per 1M tokens
Cached input: $0.075 per 1M tokens
Output: $4.50 per 1M tokens

Using those prices, the cost for a single call is:

Input cost = (input_tokens / 1 million) × 0.75

Output cost = (output_tokens / 1 million) × 4.50

Total cost = input cost + output cost

You can ignore cached input for now unless you explicitly use prompt caching; for your course project just treat every token as “standard” input.

Your notebook dataframe can now compare not just relevance and groundedness, but also:

answer_total_cost_usd

eval_total_cost_usd

overall_total_cost_usd

That will make it much easier to discuss the tradeoff between retrieval mode quality and LLM cost.



In [53]:
df["total_cost_usd"].sum()
df.groupby("search_type")["total_cost_usd"].sum()

KeyError: 'total_cost_usd'

In [30]:
df["total_cost_usd"].sum()
df.groupby("search_type")["total_cost_usd"].sum()

KeyError: 'total_cost_usd'

Given your current notebook setup and the way you’ve built the indices, you’re actually very close to the kind of ground-truth design I suggested earlier. Let’s tie it back to that “three-level” idea and make it concrete for your project.

You’re working with synthetic, Synthea-style longitudinal EHR, which has rich structured data and many documents per patient. In that setting, treating each entire patient record as a single “document with 5 questions” is too coarse; instead, it’s more useful to define gold data at three levels:

1. retrieval: which chunks should be retrieved,  
2. evidence: which specific rows/sections support an answer,  
3. final summary: the question–answer pair itself.

### 1. Retrieval-level ground truth

This is about: “For this question, which chunks should a good retriever bring back (at least one of)?”

Given how you’ve normalized the data, a retrieval-level gold item for one question might look like:

- `patient_id`: a precise patient identifier (like the ones you’re using in the notebook).
- `question`: e.g., “What oncology-related events are documented?”
- `gold_chunk_ids`: a small set of `chunk_id`s (or doc_type + heading combinations) that you consider “must retrieve” or “ideal to retrieve.”

In practice for your capstone:

- For each patient-question pair, manually identify a few chunks that clearly contain the answer (e.g., the oncology timeline entries, diagnostic reports, or problem list entries).
- Record their `chunk_id`, `doc_type`, and maybe `heading`.

Then retrieval evaluation becomes:

- Hit@k: Does any `gold_chunk_id` appear in the top k results for lexical, semantic, or hybrid?
- Mean Rank or MRR: How high are those gold chunks ranked?

You already have `search_results` in your `rag()` output; so from an evaluation perspective you can:

- compare `search_results` against your `gold_chunk_ids`,
- and compute simple metrics per retrieval mode.

This is the “retrieval” level of ground truth.

### 2. Evidence-level ground truth

This is more fine-grained: “Exactly which source elements justify the answer?”

For your project, evidence-level gold could be defined as:

- the exact table rows (e.g., in `oncology_timeline_events.csv`),
- or specific rows in `conditions.csv`, `procedures.csv`, `diagnostic_reports.csv`,
- plus any relevant fields like date, code, and text.

A single evidence item might be:

- `patient_id`
- `question`
- `evidence_rows`: list of `(table_name, row_id)` or `(doc_type, resource_id)` pairs

For example:

- `(doc_type="DiagnosticReport", resource_id="dr-123", date="2022-03-01", label="Biopsy result showing carcinoma")`
- `(doc_type="Condition", resource_id="cond-456", label="Breast cancer", recorded_date="2021-11-10")`

These are the specific “facts” your answer is supposed to rely on.

In the notebook, you can:

- inspect `search_results` for each mode,
- check whether the retrieved chunks correspond to those `evidence_rows`,
- and mark whether the evidence is present, partially present, or missing.

This connects directly to your `Groundedness` evaluation: an answer is fully grounded if the key evidence rows you defined are present in the retrieved context and correctly reflected in the answer text.

### 3. Final summary (question–answer gold)

This is the level you already started approximating with your LLM evaluation: “For this patient and this question, what is the ideal answer?”

For each patient and question, you define:

- `patient_id`
- `question`
- `gold_answer`: a concise natural-language answer that the RAG system should produce.
- optionally, a short rubric: what must be mentioned (e.g., “mention all major oncology events with dates”).

This is the question–answer pair you can compare your model’s answer against. In your current setup:

- you’re using an LLM-as-a-judge to classify answer relevance and groundedness.
- for a small curated set, you can also read the answers yourself and decide if they match the gold answer (or are “close enough”).

Taken together:

- **retrieval-level gold**: which chunks should be retrieved;
- **evidence-level gold**: which specific source rows justify the answer;
- **summary-level gold**: what the answer should say.

For your capstone you don’t need a huge dataset, but even a small table like:

- 2–3 patients,
- 3–5 questions per patient,
- and for each question a list of gold chunks, gold evidence rows, and a gold answer

would clearly demonstrate that you understand how to design RAG evaluation on structured EHR.

### How this fits with your notebook work

What you’re already doing in the notebook is:

- running questions across lexical / semantic / hybrid,
- capturing `search_results`, answers, relevance and groundedness.

To layer in the “three-level” ground truth design:

1. Pick a few patient-question pairs you care about.
2. For each, manually:
   - identify the “must-have” chunks (`gold_chunk_ids`),
   - identify one or more key source rows/entries (`evidence_rows`),
   - write a short gold answer.

3. Then, in the notebook:
   - check if each retrieval mode hits any `gold_chunk_id` in its top-k,
   - check if the context for the answer includes your `evidence_rows`,
   - compare the generated answer to your gold answer (human judgment or LLM-as-a-judge).

That will make your evaluation much more concrete and tied to the patient-level understanding you built in the course, without requiring a big automated benchmark.

If you tell me how many patients and questions you realistically want to include in your ground-truth set, I can suggest a very small, practical schema for storing this gold data (e.g., a single CSV with columns for patient_id, question, gold_answer, and a JSON field containing gold_chunk_ids and evidence rows).